# 01d — Dọn 2 mục còn treo của nhánh bệnh lá

Notebook này gồm 2 phần độc lập, chỉ **chẩn đoán, không sửa dataset**:

**Phần A — Audit chất lượng `leaf_zenodo`** để có cơ sở quyết định gộp hay không (theo 4 tiêu chí ở `HUONG_DAN_XAY_DUNG_DATASET_SMART_GREENHOUSE_AI.md` mục 2.2): class mapping rõ ràng, label đạt chất lượng, không trùng lặp nghiêm trọng, không gây leakage. `zenodo_audit.md` đã có sẵn số liệu số lượng + trùng lặp SHA-256 (0%); phần còn thiếu là **xem trực quan chất lượng label** — chưa từng làm.

**Phần B — Xem 26 ảnh negative nhóm ưu tiên 1** (`priority == 1` trong `negative_images_full_scan.csv` của `01c_review_negative_leaf.ipynb`) — chỉ bị nghi ngờ bởi **1 trong 2** lớp lọc (từ khóa tên file HOẶC model, không phải cả hai), nên chưa đủ bằng chứng để tự động loại như nhóm ưu tiên 2 đã xử lý — cần xem bằng mắt.

## Trước khi chạy
1. **Add Input** → output đã Save Version của `00_data_acquisition_leaf.ipynb` (chứa `raw/leaf_zenodo/`, cho Phần A).
2. **Add Input** thêm → output đã Save Version mới nhất của `01_build_tomato_leaf_disease_v1.ipynb` (dataset `tomato_leaf_disease_v1` đã loại 62 ảnh xác nhận, cho Phần B).
3. **Add Input** thêm → output đã Save Version của `01c_review_negative_leaf.ipynb` (chứa `negative_images_full_scan.csv`, cho Phần B).
4. Không cần GPU. Internet: OFF được.

In [ ]:
import random
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import pandas as pd
import yaml
from PIL import Image

random.seed(42)

# Phần A — Audit chất lượng `leaf_zenodo`

### Bước A1 — Tự nhận diện input

In [ ]:
ZENODO_DIR = None
for p in Path("/kaggle/input").rglob("leaf_zenodo"):
    if p.is_dir() and (p / "data.yaml").exists():
        ZENODO_DIR = p
        break

if ZENODO_DIR is None:
    print("Các thư mục cấp 1 trong /kaggle/input:")
    for p in sorted(Path("/kaggle/input").glob("*")):
        print(" -", p)
    raise FileNotFoundError(
        "Không tìm thấy raw/leaf_zenodo trong /kaggle/input. Kiểm tra đã Add Input output đã Save "
        "Version của 00_data_acquisition_leaf.ipynb chưa."
    )

zenodo_yaml = yaml.safe_load((ZENODO_DIR / "data.yaml").read_text(encoding="utf-8"))
ZENODO_CLASSES = zenodo_yaml["names"]
print("ZENODO_DIR:", ZENODO_DIR)
print("Class:", ZENODO_CLASSES)

# Gộp toàn bộ ảnh+label của cả 3 split gốc (train/valid/test) lại để audit chung, không quan tâm
# split gốc của Zenodo vì dataset này chưa quyết định gộp hay không.
zenodo_images = sorted(ZENODO_DIR.rglob("*.jpg")) + sorted(ZENODO_DIR.rglob("*.JPG"))
print(f"Tổng số ảnh: {len(zenodo_images)}")

### Bước A2 — Phân bố class thật (đếm trực tiếp từ label, không dựa vào mô tả)

In [ ]:
def label_path_for(img_path):
    return img_path.parent.parent / "labels" / (img_path.stem + ".txt")


def read_yolo_boxes(label_path):
    if not label_path.exists():
        return []
    boxes = []
    for line in label_path.read_text().splitlines():
        parts = line.split()
        if len(parts) < 5:
            continue
        cid = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        boxes.append((cid, xc, yc, bw, bh))
    return boxes


from collections import defaultdict

class_counts = defaultdict(int)
n_negative = 0
by_class_images = defaultdict(list)

for img_path in zenodo_images:
    boxes = read_yolo_boxes(label_path_for(img_path))
    if not boxes:
        n_negative += 1
        continue
    seen = set()
    for cid, *_ in boxes:
        class_counts[ZENODO_CLASSES[cid]] += 1
        seen.add(cid)
    for cid in seen:
        by_class_images[cid].append(img_path)

dist_df = pd.DataFrame([{"class": c, "n_boxes": n} for c, n in class_counts.items()]).sort_values(
    "n_boxes", ascending=False)
print(dist_df.to_string(index=False))
print(f"\nẢnh negative (không box): {n_negative}")

### Bước A3 — Xem trực quan mẫu mỗi class (đánh giá chất lượng box, chưa từng làm trước đây)
Đây là phần thông tin còn thiếu duy nhất trong 4 tiêu chí gộp — kiểm tra box có khoanh đúng/khít vùng bệnh không.

In [ ]:
def show_class_samples(cid, n=3):
    imgs = by_class_images.get(cid, [])
    if not imgs:
        return
    sample = random.sample(imgs, min(n, len(imgs)))
    fig, axes = plt.subplots(1, len(sample), figsize=(4 * len(sample), 4))
    axes = [axes] if len(sample) == 1 else axes
    for ax, img_path in zip(axes, sample):
        img = Image.open(img_path)
        w, h = img.size
        ax.imshow(img)
        for bcid, xc, yc, bw, bh in read_yolo_boxes(label_path_for(img_path)):
            if bcid != cid:
                continue
            xmin, ymin = (xc - bw / 2) * w, (yc - bh / 2) * h
            rect = patches.Rectangle((xmin, ymin), bw * w, bh * h, linewidth=2, edgecolor="lime", facecolor="none")
            ax.add_patch(rect)
        ax.set_title(img_path.name, fontsize=6)
        ax.axis("off")
    fig.suptitle(ZENODO_CLASSES[cid], fontsize=10)
    plt.tight_layout()
    plt.savefig(f"/kaggle/working/zenodo_sample_{ZENODO_CLASSES[cid]}.png", dpi=90, bbox_inches="tight")
    plt.show()
    plt.close(fig)


for cid in sorted(by_class_images.keys()):
    show_class_samples(cid)

### Kết luận Phần A — đối chiếu 4 tiêu chí gộp
- [ ] **Class ánh xạ rõ ràng**: 9/10 class khớp trực tiếp — `Tomato_Yellow_Leaf_Curl_Virus`→OUT_OF_SCOPE, `Tomato__Spider_Mites`→OUT_OF_SCOPE, `Tomato___Bacterial_spot`→OUT_OF_SCOPE, `Tomato___Early_blight`→`leaf_early_blight`, `Tomato___Late_blight`→`leaf_late_blight`, `Tomato___Leaf_Mold`→`leaf_mold`, `Tomato___Septoria_leaf_spot`→`leaf_septoria_spot`, `Tomato_mosaic_virus`→OUT_OF_SCOPE, `healthy_leaf`→NEGATIVE. Riêng `Tomato___Target_Spot` **không có** trong 11 class của `leaf_roboflow` — cần quyết định: OUT_OF_SCOPE (loại, an toàn) hay coi là bệnh khác (không nằm trong phạm vi MVP hiện tại → nên loại).
- [ ] **Chất lượng label**: xem kết quả Bước A3 ở trên bằng mắt trước khi đánh dấu mục này.
- [x] **Không trùng lặp nghiêm trọng (mức sàn)**: đã xác nhận từ `01_build_tomato_leaf_disease_v1.ipynb` — 0/23.261 ảnh trùng SHA-256 chính xác với `leaf_roboflow`. Chưa kiểm tra near-duplicate (pHash) chéo.
- [ ] **Không gây leakage nếu gộp**: chỉ đánh giá được sau khi thực sự gộp + dedup lại toàn bộ (áp dụng lại đúng quy trình Bước 5 của `01_build_tomato_leaf_disease_v1.ipynb` cho cả 2 nguồn cùng lúc).

---

# Phần B — Xem 26 ảnh negative nhóm ưu tiên 1

### Bước B1 — Tự nhận diện input + lọc `priority == 1`

In [ ]:
TARGET_CLASSES = ["leaf_early_blight", "leaf_late_blight", "leaf_mold", "leaf_septoria_spot"]


def find_leaf_dataset_root():
    for yf in Path("/kaggle/input").rglob("data.yaml"):
        try:
            y = yaml.safe_load(yf.read_text(encoding="utf-8"))
        except Exception:
            continue
        names = y.get("names") if isinstance(y, dict) else None
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names == TARGET_CLASSES:
            return yf.parent
    return None


LEAF_DATASET_ROOT = find_leaf_dataset_root()
scan_csv_hits = list(Path("/kaggle/input").rglob("negative_images_full_scan.csv"))

if LEAF_DATASET_ROOT is None:
    raise FileNotFoundError(
        "Không tìm thấy data.yaml khớp 4 class của tomato_leaf_disease_v1. Kiểm tra Add Input."
    )
if not scan_csv_hits:
    raise FileNotFoundError(
        "Không tìm thấy negative_images_full_scan.csv. Kiểm tra đã Add Input output đã Save Version "
        "của 01c_review_negative_leaf.ipynb chưa."
    )

print("LEAF_DATASET_ROOT:", LEAF_DATASET_ROOT)
print("SCAN_CSV         :", scan_csv_hits[0])

scan_df = pd.read_csv(scan_csv_hits[0])
priority1_df = scan_df[scan_df["priority"] == 1].copy()
print(f"\nSố ảnh negative nhóm ưu tiên 1: {len(priority1_df)}")
print(priority1_df[["orig_filename", "split", "keyword_flag", "model_flag", "model_top_class", "model_top_conf"]]
      .to_string(index=False))

### Bước B2 — Xác nhận trực quan
Cột `path` trong CSV là đường dẫn tuyệt đối từ session Kaggle cũ của `01c` — không còn tồn tại ở đây. Lấy phần **tên file** cuối của `path` (không đổi giữa các session) rồi tìm lại trong dataset hiện tại, tìm khắp `train/val/test` vì split có thể đã thay đổi sau khi build lại.

In [ ]:
def find_image_anywhere(filename):
    for split in ["train", "val", "test"]:
        p = LEAF_DATASET_ROOT / split / "images" / filename
        if p.exists():
            return p, split
    return None, None


priority1_df["filename"] = priority1_df["path"].apply(lambda p: Path(p).name)
found_rows = []
for _, row in priority1_df.iterrows():
    img_path, cur_split = find_image_anywhere(row["filename"])
    if img_path is None:
        print(f"[không tìm thấy] {row['filename']} (có thể đã bị loại ở lần build lại)")
        continue
    found_rows.append({**row.to_dict(), "current_path": img_path, "current_split": cur_split})

found_df = pd.DataFrame(found_rows)
print(f"\nTìm thấy {len(found_df)}/{len(priority1_df)} ảnh trong dataset hiện tại.")

n = len(found_df)
cols = 6
rows_n = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_n, cols, figsize=(3.2 * cols, 3.2 * rows_n))
axes = axes.flatten() if rows_n * cols > 1 else [axes]
for ax, (_, row) in zip(axes, found_df.iterrows()):
    img = Image.open(row["current_path"])
    ax.imshow(img)
    flag_desc = []
    if row["keyword_flag"]:
        flag_desc.append(f"tên_file:{row['keyword_hits']}")
    if row["model_flag"]:
        flag_desc.append(f"model:{row['model_top_class']}({row['model_top_conf']:.2f})")
    ax.set_title(f"{row['orig_filename'][:22]}\n{' | '.join(flag_desc)}", fontsize=6)
    ax.axis("off")
for ax in axes[len(found_df):]:
    ax.axis("off")
fig.suptitle("Negative ưu tiên 1 — chỉ 1 trong 2 lớp lọc nghi ngờ", fontsize=11)
plt.tight_layout()
plt.savefig("/kaggle/working/priority1_negative_review.png", dpi=90, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
found_df.drop(columns=["current_path"]).to_csv("/kaggle/working/priority1_negative_review.csv", index=False)
print("Đã lưu: /kaggle/working/priority1_negative_review.csv")

## Kết quả & cách dùng
- **Phần A**: nếu ảnh mẫu ở Bước A3 cho thấy box khoanh đúng/khít vùng bệnh (giống chất lượng `leaf_roboflow`) và bạn đồng ý loại `Target_Spot`, có thể quyết định gộp `leaf_zenodo` — khi đó cần viết thêm bước gộp vào `01_build_tomato_leaf_disease_v1.ipynb` (thêm `leaf_zenodo` vào `CLASS_MAPPING`, chạy lại dedup/split cho cả 2 nguồn cùng lúc). Nếu chất lượng box kém hoặc lệch nhiều so với `leaf_roboflow`, giữ nguyên quyết định "chưa gộp" hiện tại.
- **Phần B**: `priority1_negative_review.csv` + ảnh ở Bước B2 — ảnh nào bạn xác nhận thực sự có dấu hiệu bệnh, ghi lại `orig_filename` để bổ sung thủ công vào danh sách loại trừ (cùng cơ chế `review_candidates_negative.csv` đã dùng cho nhóm ưu tiên 2) ở lần build tiếp theo.